In [ ]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from shapely import make_valid
from shapely.geometry import mapping


# ============================================================
# 1. Configuration
# ============================================================

BASE = Path("../data")
DEM_ROOT = Path("../data/raw/dem")

SMOD = BASE / "smod_0514.shp"
PANEL = Path("../data/study_grid_cells.csv")


OUT_BASE = Path(
    "../outputs/terrain"
)

RES = 30.0
COVERAGE_MIN = 0.90

MARGIN = 1000.0

YEARS = list(range(2016, 2024))
KEY = ["city", "cell_id"]

CONFIG = {
    "Abidjan_IvoryCoast":
        ("Abidjan", "Abidjan", 32630),
    "AddisAbaba_Ethiopia":
        ("Addis_Ababa", "Addis Ababa", 32637),
    "Alexandria_Egypt":
        ("Alexandria", "Alexandria", 32635),
    "Bamako_Mali":
        ("Bamako", "Bamako", 32629),
    "Bangalore_India":
        ("Bangalore", "Bangalore", 32643),
    "Bangkok_Thailand":
        ("Bangkok", "Bangkok", 32647),
    "BeloHorizonte_Brazil":
        ("Belo_Horizonte", "Belo Horizonte", 32723),
    "Bogota_Colombia":
        ("Bogota", "Bogota", 32618),
    "BuenosAires_Argentina":
        ("Buenos_Aires", "Buenos Aires", 32721),
    "Cairo_Egypt":
        ("Cairo", "Cairo", 32636),
    "Chennai_India":
        ("Chennai", "Chennai", 32644),
    "Colombo_SriLanka":
        ("Colombo", "Colombo", 32644),
    "DaresSalaam_Tanzania":
        ("Dar_es_Salaam", "Dar es Salaam", 32737),
    "Delhi_India":
        ("Delhi", "Delhi", 32643),
    "Dhaka_Bangladesh":
        ("Dhaka", "Dhaka", 32646),
    "Guayaquil_Ecuador":
        ("Guayaquil", "Guayaquil", 32717),
    "HoChiMinh_Vietnam":
        ("Ho_Chi_Minh_City", "Ho Chi Minh City", 32648),
    "Hyderabad_India":
        ("Hyderabad", "Hyderabad", 32644),
    "Jakarta_Indonesia":
        ("Jakarta", "Jakarta", 32748),
    "Johannesburg_SouthAfrica":
        ("Johannesburg", "Johannesburg", 32735),
    "Karachi_Pakistan":
        ("Karachi", "Karachi", 32642),
    "Kinshasa_DRC":
        ("Kinshasa", "Kinshasa", 32733),
    "KualaLumpur_Malaysia":
        ("Kuala_Lumpur", "Kuala Lumpur", 32647),
    "Lagos_Nigeria":
        ("Lagos", "Lagos", 32631),
    "Lahore_Pakistan":
        ("Lahore", "Lahore", 32643),
    "Lima_Peru":
        ("Lima", "Lima", 32718),
    "Manaus_Brazil":
        ("Manaus", "Manaus", 32721),
    "Manila_Philippines":
        ("Manila", "Manila", 32651),
    "MexicoCity_Mexico":
        ("Mexico_City", "Mexico City", 32614),
    "Mumbai_India":
        ("Mumbai", "Mumbai", 32643),
    "Nairobi_Kenya":
        ("Nairobi", "Nairobi", 32737),
    "PhnomPenh_Cambodia":
        ("Phnom_Penh", "Phnom Penh", 32648),
    "Santiago_Chile":
        ("Santiago", "Santiago", 32719),
    "SantoDomingo_DominicanRepublic":
        ("Santo_Domingo", "Santo Domingo", 32619),
    "SaoPaulo_Brazil":
        ("Sao_Paulo", "Sao Paulo", 32723),
}


# ============================================================
# 2. Basic checks
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def stamp(path):
    s = Path(path).stat()
    return dict(
        bytes=s.st_size,
        mtime_ns=s.st_mtime_ns,
    )


def valid_polygon(geom):
    require(
        geom is not None and not geom.is_empty,
        "Empty geometry",
    )

    if geom.is_valid:
        return geom

    fixed = make_valid(geom)

    require(
        fixed.geom_type in ["Polygon", "MultiPolygon"]
        and fixed.is_valid
        and abs(fixed.area - geom.area)
        <= max(1e-12, abs(geom.area) * 1e-9)
        and np.allclose(
            fixed.bounds,
            geom.bounds,
            atol=1e-10,
            rtol=0,
        ),
        "Geometry repair changed area or extent; needs separate review",
    )

    return fixed


# ============================================================
# 3. Slope
# ============================================================

def slope_horn(z):
    result = np.full(
        z.shape, np.nan, dtype="float64"
    )

    a = z[:-2, :-2]
    b = z[:-2, 1:-1]
    c = z[:-2, 2:]

    d = z[1:-1, :-2]
    e = z[1:-1, 1:-1]
    f = z[1:-1, 2:]

    g = z[2:, :-2]
    h = z[2:, 1:-1]
    i = z[2:, 2:]

    valid = np.logical_and.reduce([
        np.isfinite(v)
        for v in (a, b, c, d, e, f, g, h, i)
    ])

    dx = (
        (c + 2*f + i) - (a + 2*d + g)
    ) / (8 * RES)

    dy = (
        (g + 2*h + i) - (a + 2*b + c)
    ) / (8 * RES)

    values = np.degrees(
        np.arctan(np.hypot(dx, dy))
    )

    result[1:-1, 1:-1] = np.where(
        valid, values, np.nan
    )

    return result


# ============================================================
# 4. Project to the 30 m grid, keeping the neighbourhood outside the boundary
# ============================================================

def project_window(
    z,
    support,
    source_transform,
    source_crs,
    epsg,
    target,
):
    x0, y0, x1, y1 = target.bounds

    pad = MARGIN + 3 * RES

    left = math.floor((x0 - pad) / RES) * RES
    bottom = math.floor((y0 - pad) / RES) * RES
    right = math.ceil((x1 + pad) / RES) * RES
    top = math.ceil((y1 + pad) / RES) * RES

    w = int(round((right - left) / RES))
    h = int(round((top - bottom) / RES))

    require(
        w * h <= 20_000_000,
        "Computation window abnormally large; check geometry/projection",
    )

    transform = from_origin(
        left, top, RES, RES
    )

    dst = np.full(
        (h, w), np.nan, dtype="float32"
    )

    fraction = np.zeros(
        (h, w), dtype="float32"
    )

    args = dict(
        src_transform=source_transform,
        src_crs=source_crs,
        dst_transform=transform,
        dst_crs=f"EPSG:{epsg}",
        resampling=Resampling.bilinear,
        num_threads=2,
    )

    reproject(
        z,
        dst,
        src_nodata=np.nan,
        dst_nodata=np.nan,
        **args,
    )

    reproject(
        support,
        fraction,
        src_nodata=None,
        dst_nodata=0,
        **args,
    )

    dst[fraction < 1 - 1e-6] = np.nan

    inside = geometry_mask(
        [mapping(target)],
        out_shape=(h, w),
        transform=transform,
        invert=True,
        all_touched=False,
    )

    return dst.astype("float64"), inside


# ============================================================
# 5. Summarise metrics and valid-data coverage separately
# ============================================================

def summarize(
    rec,
    name,
    values,
    inside,
    mean_col,
    std_col=None,
):
    vals = values[
        inside & np.isfinite(values)
    ]

    n = int(inside.sum())
    nv = int(vals.size)

    coverage = nv / n if n else np.nan

    passed = bool(
        n and coverage >= COVERAGE_MIN
    )

    rec[name + "_valid_pixels"] = nv
    rec[name + "_coverage_fraction"] = coverage
    rec[name + "_coverage_pass"] = passed

    rec[mean_col + "_observed"] = (
        float(vals.mean()) if nv else np.nan
    )

    rec[mean_col] = (
        rec[mean_col + "_observed"]
        if passed else np.nan
    )

    if std_col:
        rec[std_col + "_observed"] = (
            float(vals.std(ddof=0))
            if nv else np.nan
        )

        rec[std_col] = (
            rec[std_col + "_observed"]
            if passed else np.nan
        )


# ============================================================
# 6. Main program
# ============================================================

def run_terrain():
    require(
        RES > 0,
        "Resolution configuration error",
    )

    inputs = []

    def register(path, role):
        require(
            path.is_file(),
            f"Missing input: {path}",
        )

        inputs.append(
            dict(
                path=str(path),
                role=role,
                **stamp(path),
            )
        )

    register(PANEL, "study_point_list")

    for ext in [".shp", ".shx", ".dbf", ".prj"]:
        register(
            SMOD.with_suffix(ext),
            "smod_boundary",
        )

    p = pd.read_csv(PANEL)

    points = p[
        ["city", "cell_id", "lat", "lon"]
    ].drop_duplicates()

    require(
        len(points) == 652
        and points.city.nunique() == 35
        and set(points.city) == set(CONFIG)
        and not points.duplicated(KEY).any(),
        "Study list is not the expected 35 cities and 652 grid cells",
    )

    b = gpd.read_file(SMOD)

    require(
        b.crs is not None,
        "SMOD has no CRS",
    )

    b = b.to_crs(4326)
    jobs = []

    # --------------------------------------------------------
    # Pre-checks: all inputs, geometries, fixed study grid cells
    # --------------------------------------------------------

    for city, (prefix, name, epsg) in CONFIG.items():
        path = (
            DEM_ROOT / f"{prefix}_DEM_GLO30.tif"
        )

        gp = (
            BASE / "grids"
            / f"{city}_era5_grid.gpkg"
        )

        register(path, "dem")
        register(gp, "fixed_research_grid")

        with rasterio.open(path) as src:
            require(
                src.count == 1
                and src.crs is not None,
                f"DEM band/CRS abnormal: {path}",
            )

            require(
                src.scales[0] == 1
                and src.offsets[0] == 0,
                f"DEM scaling abnormal: {path}",
            )

        sub = b[b["name"].eq(name)]

        require(
            len(sub) == 1,
            f"SMOD cannot be matched uniquely: {city}",
        )

        urban = valid_polygon(
            sub.geometry.iloc[0]
        )

        grid = gpd.read_file(gp)

        require(
            grid.crs is not None
            and grid.cell_id.is_unique,
            f"Abnormal grid: {city}",
        )

        grid = (
            grid.to_crs(4326)
            .set_index("cell_id")
        )

        selected = (
            points[points.city.eq(city)]
            .sort_values("cell_id")
        )

        records = []
        targets = []

        for row in selected.itertuples():
            g = grid.loc[row.cell_id]

            require(
                np.allclose(
                    [row.lat, row.lon],
                    [g.lat_center, g.lon_center],
                    atol=1e-6,
                    rtol=0,
                ),
                f"Grid-cell coordinates do not match: {city} {row.cell_id}",
            )

            target = valid_polygon(
                g.geometry
            ).intersection(urban)

            require(
                not target.is_empty
                and target.area > 0,
                f"Empty study area: {city} {row.cell_id}",
            )

            targets.append(target)

            records.append(
                dict(
                    city=city,
                    cell_id=int(row.cell_id),
                    lat=float(row.lat),
                    lon=float(row.lon),
                )
            )

        projected = gpd.GeoSeries(
            targets,
            crs=4326,
        ).to_crs(epsg)

        jobs.append(
            (
                city,
                path,
                epsg,
                records,
                list(projected),
            )
        )

    # --------------------------------------------------------
    # Create a new output folder
    # --------------------------------------------------------

    out = OUT_BASE

    (out / "by_city").mkdir(
        parents=True,
        exist_ok=True,
    )

    summary = dict(
        status="running",
        resolution_m=RES,
        coverage_threshold=COVERAGE_MIN,
        units=dict(
            elevation="m",
            slope="degrees",
        ),
    )

    def save_summary():
        (out / "summary.json").write_text(
            json.dumps(
                summary,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    def save(frame, name):
        frame = frame.reset_index(drop=True)
        file = out / name

        frame.to_csv(
            file,
            index=False,
            encoding="utf-8-sig",
            float_format="%.17g",
        )

        back = pd.read_csv(
            file,
            float_precision="round_trip",
        )

        pd.testing.assert_frame_equal(
            frame,
            back,
            check_dtype=False,
            check_exact=True,
        )

    save_summary()

    pd.DataFrame(inputs).to_csv(
        out / "input_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

    all_rows = []
    source_rows = []

    print("Output folder:", out, flush=True)

    try:
        # ----------------------------------------------------
        # Read the DEM once per city and process grid cell by grid cell
        # ----------------------------------------------------

        for city, path, epsg, records, targets in jobs:
            for attempt in range(3):
                try:
                    with rasterio.open(path) as src:
                        arr = src.read(
                            1, masked=True
                        )
                        st = src.transform
                        crs = src.crs
                    break

                except (
                    rasterio.errors.RasterioIOError,
                    OSError,
                ):
                    if attempt == 2:
                        raise
                    time.sleep(3)

            v = np.asarray(
                arr.data,
                dtype="float32",
            )

            valid = (
                ~np.ma.getmaskarray(arr)
                & np.isfinite(v)
                & (v != -9999)
            )

            source_rows.append(
                dict(
                    city=city,
                    source_path=str(path),
                    source_crs=str(crs),
                    rows=v.shape[0],
                    cols=v.shape[1],
                    valid_pixels=int(valid.sum()),
                    missing_pixels=int(
                        (~valid).sum()
                    ),
                    negative_valid_pixels=int(
                        (valid & (v < 0)).sum()
                    ),
                )
            )

            z = np.where(
                valid, v, np.nan
            ).astype("float32")

            support = valid.astype("float32")

            del arr, v, valid

            rows = []

            for info, target in zip(records, targets):
                dem, inside = project_window(
                    z,
                    support,
                    st,
                    crs,
                    epsg,
                    target,
                )

                rec = dict(
                    info,
                    source_path=str(path),
                    utm_epsg=epsg,
                    study_area_m2=float(target.area),
                    study_pixels=int(inside.sum()),
                    rasterized_study_area_m2=(
                        int(inside.sum()) * RES**2
                    ),
                )

                summarize(
                    rec,
                    "elevation",
                    dem,
                    inside,
                    "dem_mean_m",
                    "dem_std_m",
                )

                slope = slope_horn(dem)

                summarize(
                    rec,
                    "slope",
                    slope,
                    inside,
                    "dem_mean_slope",
                )

                rec["slope_neighbor_unavailable_pixels"] = (
                    rec["elevation_valid_pixels"]
                    - rec["slope_valid_pixels"]
                )

                vals = dem[
                    inside & np.isfinite(dem)
                ]

                rec[
                    "elev_mean_minus_median_observed"
                ] = (
                    float(
                        vals.mean() - np.median(vals)
                    )
                    if vals.size else np.nan
                )

                rec["terrain_main_complete"] = all(
                    rec[k + "_coverage_pass"]
                    for k in [
                        "elevation",
                        "slope",
                    ]
                )

                rec["needs_review"] = (
                    any(
                        rec[k + "_valid_pixels"]
                        < rec["study_pixels"]
                        for k in [
                            "elevation",
                            "slope",
                        ]
                    )
                    or rec["study_pixels"] == 0
                )

                rec["terrain_imputed"] = False
                rows.append(rec)

                print(
                    f"[{len(all_rows) + len(rows)}/652] "
                    f"{city} cell={info['cell_id']}: "
                    f"elevation "
                    f"{rec['elevation_coverage_fraction']:.1%}, "
                    f"slope "
                    f"{rec['slope_coverage_fraction']:.1%}",
                    flush=True,
                )

            save(
                pd.DataFrame(rows),
                f"by_city/{city}.csv",
            )

            all_rows.extend(rows)

            summary.update(
                points_finished=len(all_rows)
            )
            save_summary()

            del z, support

        # ----------------------------------------------------
        # Merge and validate
        # ----------------------------------------------------

        df = (
            pd.DataFrame(all_rows)
            .sort_values(KEY)
            .reset_index(drop=True)
        )

        require(
            len(df) == 652
            and not df.duplicated(KEY).any(),
            "Output grid-cell count or keys abnormal",
        )

        checks = [
            (
                "elevation",
                ["dem_mean_m", "dem_std_m"],
            ),
            (
                "slope",
                ["dem_mean_slope"],
            ),
        ]

        for kind, cols in checks:
            passed = df[kind + "_coverage_pass"]

            require(
                df.loc[
                    ~passed, cols
                ].isna().all().all(),
                f"{kind}: low-coverage values wrongly kept",
            )

            require(
                np.isfinite(
                    df.loc[
                        passed, cols
                    ].to_numpy(float)
                ).all(),
                f"{kind}: values meeting the coverage threshold are abnormal",
            )

            require(
                df[kind + "_valid_pixels"]
                .le(df.study_pixels).all(),
                "Valid pixels exceed study-area pixels",
            )

        require(
            df.dem_mean_slope_observed
            .dropna().between(0, 90).all(),
            "Slope outside 0–90 degrees",
        )

        for r in inputs:
            require(
                stamp(r["path"]) == {
                    k: r[k]
                    for k in ["bytes", "mtime_ns"]
                },
                f"Input changed during computation: {r['path']}",
            )

        annual = (
            df.merge(
                pd.DataFrame({"year": YEARS}),
                how="cross",
            )
            .sort_values(KEY + ["year"])
        )

        require(
            len(annual) == 5216
            and not annual.duplicated(
                KEY + ["year"]
            ).any(),
            "Annual table row count abnormal",
        )

        save(df, "terrain_smod_static.csv")

        save(
            annual,
            "terrain_smod_2016_2023.csv",
        )

        save(
            df[~df.terrain_main_complete],
            "terrain_below90.csv",
        )

        save(
            df[df.needs_review],
            "terrain_rows_requiring_review.csv",
        )

        save(
            pd.DataFrame(source_rows),
            "dem_source_checks.csv",
        )

        qc = df.groupby(
            "city", as_index=False
        ).agg(
            points=("cell_id", "size"),
            elevation_pass=(
                "elevation_coverage_pass", "sum"
            ),
            slope_pass=(
                "slope_coverage_pass", "sum"
            ),
            main_complete=(
                "terrain_main_complete", "sum"
            ),
        )

        save(qc, "city_quality.csv")

        summary.update(
            status="calculation_complete",
            static_rows=len(df),
            annual_rows=len(annual),
            cities=35,
            elevation_pass=int(
                df.elevation_coverage_pass.sum()
            ),
            slope_pass=int(
                df.slope_coverage_pass.sum()
            ),
            terrain_main_complete=int(
                df.terrain_main_complete.sum()
            ),
        )

        save_summary()

    except (Exception, KeyboardInterrupt) as exc:
        summary.update(
            status="failed_or_interrupted",
            error=repr(exc),
            points_finished=len(all_rows),
        )
        save_summary()
        raise

    print("Output folder:", out)

    return out, df


TERRAIN_SMOD_DIR, TERRAIN_SMOD_RESULTS = run_terrain()